Notebook for Becode workshop, Second, bakehouse and free tier testing

sample datasets are 'read only', we create a new db

In [0]:
%sql
SELECT current_catalog(), current_schema();
CREATE SCHEMA IF NOT EXISTS bakehouse_lab;
USE SCHEMA bakehouse_lab;

## Exploring dataset


In [0]:
%sql
SELECT 'sales_transactions' AS tbl, COUNT(*) AS n FROM samples.bakehouse.sales_transactions
UNION ALL SELECT 'sales_franchises', COUNT(*) FROM samples.bakehouse.sales_franchises
UNION ALL SELECT 'sales_customers', COUNT(*) FROM samples.bakehouse.sales_customers
UNION ALL SELECT 'sales_suppliers', COUNT(*) FROM samples.bakehouse.sales_suppliers
UNION ALL SELECT 'media_customer_reviews', COUNT(*) FROM samples.bakehouse.media_customer_reviews
UNION ALL SELECT 'media_gold_reviews_chunked', COUNT(*) FROM samples.bakehouse.media_gold_reviews_chunked;

In [0]:
%sql
SELECT COUNT(*) AS orphan_transactions
FROM samples.bakehouse.sales_transactions t
LEFT JOIN samples.bakehouse.sales_franchises f ON t.franchiseID = f.franchiseID
WHERE f.franchiseID IS NULL;

## Build a silver table

In [0]:
%sql
CREATE OR REPLACE TABLE silver_transactions AS
SELECT t.transactionID, t.customerID, t.franchiseID,
       CAST(t.dateTime AS TIMESTAMP) AS purchase_ts,
       DATE(t.dateTime)              AS purchase_date,
       t.product, t.quantity, t.unitPrice, t.totalPrice, t.paymentMethod,
       f.name AS franchise_name, f.city, f.country, f.size AS franchise_size
FROM samples.bakehouse.sales_transactions t
JOIN samples.bakehouse.sales_franchises f ON t.franchiseID = f.franchiseID;

SELECT * FROM silver_transactions LIMIT 10;

## Build a Gold table

In [0]:
%sql
CREATE OR REPLACE TABLE gold_franchise_revenue AS
SELECT franchiseID, franchise_name, city, country,
       COUNT(*)        AS n_transactions,
       SUM(quantity)   AS units_sold,
       SUM(totalPrice) AS revenue
FROM silver_transactions
GROUP BY franchiseID, franchise_name, city, country;

SELECT * FROM gold_franchise_revenue ORDER BY revenue DESC LIMIT 10;

## Add AI:  review sentiment

In [0]:
%sql
CREATE OR REPLACE TABLE silver_review_sentiment AS
SELECT franchiseID, review_date,
       ai_analyze_sentiment(review) AS sentiment,
       review
FROM samples.bakehouse.media_customer_reviews;

SELECT sentiment, COUNT(*) AS n FROM silver_review_sentiment GROUP BY sentiment;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_franchise_scorecard AS
SELECT g.franchiseID, g.franchise_name, g.country, g.revenue,
       COUNT(s.sentiment) AS n_reviews,
       ROUND(AVG(CASE WHEN s.sentiment = 'positive' THEN 1.0 ELSE 0.0 END), 2) AS share_positive
FROM gold_franchise_revenue g
LEFT JOIN silver_review_sentiment s ON g.franchiseID = s.franchiseID
GROUP BY g.franchiseID, g.franchise_name, g.country, g.revenue;

SELECT * FROM gold_franchise_scorecard ORDER BY revenue DESC;

## Specializations
### Data Analyst

In [0]:
%sql
-- 1. Revenue by product
SELECT product, SUM(totalPrice) AS revenue, SUM(quantity) AS units
FROM silver_transactions GROUP BY product ORDER BY revenue DESC;

-- 2. Revenue by country and payment method
SELECT country, paymentMethod, SUM(totalPrice) AS revenue
FROM silver_transactions GROUP BY country, paymentMethod ORDER BY country, revenue DESC;

-- 3. Daily revenue trend
SELECT purchase_date, SUM(totalPrice) AS revenue
FROM silver_transactions GROUP BY purchase_date ORDER BY purchase_date;

### Data Engineer

### Data Scientist

In [0]:
import mlflow
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = spark.sql('''
    SELECT customerID,
           COUNT(*)                AS n_purchases,
           SUM(totalPrice)         AS total_spend,
           AVG(totalPrice)         AS avg_basket,
           COUNT(DISTINCT product) AS n_products
    FROM bakehouse_lab.silver_transactions
    GROUP BY customerID
''').toPandas()

cols = ['n_purchases', 'total_spend', 'avg_basket', 'n_products']
features[cols] = features[cols].astype(float)
X = StandardScaler().fit_transform(features[cols])

mlflow.sklearn.autolog()
with mlflow.start_run(run_name='customer_segments_k3'):
    km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
    mlflow.log_metric('inertia', km.inertia_)

features['segment'] = km.labels_
print(features.groupby('segment')[cols].mean().round(1))
print(features['segment'].value_counts())

### Gen AI

In [0]:
%sql
CREATE OR REPLACE TABLE gold_review_topics AS
SELECT franchiseID, review_date,
       ai_classify(review, ARRAY('product quality', 'price', 'service', 'atmosphere', 'other')) AS main_topic,
       ai_analyze_sentiment(review) AS sentiment,
       ai_summarize(review, 25)     AS one_liner
FROM samples.bakehouse.media_customer_reviews;

SELECT main_topic, sentiment, COUNT(*) AS n
FROM gold_review_topics GROUP BY main_topic, sentiment ORDER BY n DESC;

In [0]:
%sql
SELECT * FROM gold_review_topics LIMIT 5;

In [0]:
%sql
SELECT franchiseID,
       ai_query('system.ai.databricks-gemini-3-1-flash-lite',
                CONCAT('List the cookie or pie names that this customer criticises, as a comma-separated list. If none, answer none. Review: ', review)) AS criticised_products
FROM samples.bakehouse.media_customer_reviews
LIMIT 20;